In [ ]:
import os

%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import seaborn as sns

from deephit_cancer_comparison.constants import GRAPH_PATH, SURVSHAP_PATH

TOP_K = 10

cancer_types = [
    "breast",
    "corpus",
    "kidney_parenchyma",
    "lung_and_bronchus",
    "melanoma_of_the_skin",
    "pancreas",
    "prostate",
    "thyroid",
    "urinary_bladder",
    "colon_and_rectum",
]

for cancer_type in cancer_types:
    if not os.path.exists(GRAPH_PATH / cancer_type):
        os.makedirs(GRAPH_PATH / cancer_type, exist_ok=True)

In [ ]:
FEATURE_GROUPING = {
    # pass-through (numeric or already single column)
    "age": "age",
    "year_dx": "year_dx",
    "sex": "sex",
    "summary_stage": "summary_stage",
    "tumor_size": "tumor_size",
    # flags kept as their own logical features (different info from parent)
    "stage_unknown": "stage_unknown",
    "tumor_size_unknown": "tumor_size_unknown",
    "tumor_size_no_mass": "tumor_size_no_mass",
    "marital_unknown": "marital_unknown",
    # race dummies (reference: Non-Hispanic White)
    "race_hispanic": "race",
    "race_american_indian_alaska_native": "race",
    "race_asian_pacific_islander": "race",
    "race_black": "race",
    "race_unknown": "race",
    # marital status dummies (reference: Married)
    "marital_status_Widowed": "marital_status",
    "marital_status_Divorced": "marital_status",
    "marital_status_Single (never married)": "marital_status",
    "marital_status_Separated": "marital_status",
    "marital_status_Unmarried or Domestic Partner": "marital_status",
    # histology dummies (reference: Adenocarcinomas)
    "histology_Ductal and lobular": "histology",
    "histology_Epithelial NOS": "histology",
    "histology_Squamous cell": "histology",
    "histology_Transitional cell": "histology",
    "histology_Unspecified neoplasms": "histology",
    "histology_Melanomas": "histology",
    "histology_Cystic/mucinous/serous": "histology",
    "histology_Complex epithelial": "histology",
    "histology_Complex mixed and stromal": "histology",
    "histology_Acinar cell": "histology",
    "histology_Myomatous": "histology",
    "histology_Other": "histology",
}

In [ ]:
def load_cohort(cohort: str, root: Path = SURVSHAP_PATH):
    """Load aggregated (scalar per feature per obs) and time-varying SurvSHAP results."""
    cohort_dir = root / cohort
    scalar_df = pd.read_parquet(cohort_dir / "survshap_aggregated.parquet")
    tv_df = pd.read_parquet(cohort_dir / "survshap_timevarying.parquet")
    return scalar_df, tv_df

In [ ]:
def aggregate_dummies(df, grouping):
    out = df.copy()
    out["logical_feature"] = out["variable_name"].map(grouping)

    missing = out.loc[out["logical_feature"].isna(), "variable_name"].unique()
    if len(missing):
        raise ValueError(f"FEATURE_GROUPING missing: {sorted(missing)}")

    keep_cols = [c for c in ["cohort", "iteration", "obs_id"] if c in out.columns]
    grouped = out.groupby(keep_cols + ["logical_feature"], as_index=False)[
        "aggregated_change"
    ].sum()
    return grouped

In [ ]:
def cohort_summary(per_patient_df):
    return (
        per_patient_df.groupby("logical_feature")["aggregated_change"]
        .agg(
            median="median",
            mean="mean",
            q25=lambda s: s.quantile(0.25),
            q75=lambda s: s.quantile(0.75),
            n_patients="count",
        )
        .sort_values("median", ascending=False)
    )

In [ ]:
def cross_cohort_matrix(per_patient_df, agg="median"):
    return (
        per_patient_df.groupby(["cohort", "logical_feature"])["aggregated_change"]
        .agg(agg)
        .reset_index()
        .pivot(index="logical_feature", columns="cohort", values="aggregated_change")
    )

In [ ]:
all_cohorts = []
for cancer in cancer_types:
    scalar_df, tv_df = load_cohort(cancer)
    agg_dummies = aggregate_dummies(scalar_df, FEATURE_GROUPING)
    ccm = cross_cohort_matrix(agg_dummies)
    all_cohorts.append(ccm)

df_all_cohorts = pd.concat(all_cohorts, axis=1, ignore_index=False)

In [ ]:
cancer_col_name_map = {
    "breast": "Breast",
    "corpus": "Corpus",
    "kidney_parenchyma": "Kidney Parenchyma",
    "melanoma_of_the_skin": "Melanoma",
    "lung_and_bronchus": "Lung & Bronchus",
    "pancreas": "Pancreas",
    "prostate": "Prostate",
    "thyroid": "Thyroid",
    "urinary_bladder": "Urinary Bladder",
    "colon_and_rectum": "Colorectal",
}

variable_name_map = {
    "age": "Age",
    "histology": "Histology",
    "marital_status": "Marital Status",
    "marital_unknown": "Unknown Marital Status",
    "race": "Race",
    "sex": "Sex",
    "summary_stage": "Tumor Stage",
    "stage_unknown": "Unknown Tumor Stage",
    "tumor_size": "Tumor Size",
    "tumor_size_unknown": "Unknown Tumor Size",
    "tumor_size_no_mass": "Tumor No Mass",
    "year_dx": "Year of Diagnosis",
}

In [ ]:
df_all_cohorts.rename(columns=cancer_col_name_map, index=variable_name_map, inplace=True)

In [ ]:
def plot_cancer_bar(
    matrix: pd.DataFrame,
    cancer: str,
    figsize: tuple = (10, 7),
    cmap: str = "Blues",
    cmap_minval: float = 0.3,
    cmap_maxval: float = 1.0,
    zero_color: str = "#D3D1C7",
    annot_fontsize: int = 9,
    save_path=None,
):
    def truncate_colormap(cmap, minval=0.5, maxval=1.0, n=256):
        import matplotlib.colors as mcolors
        import numpy as np

        return mcolors.LinearSegmentedColormap.from_list(
            f"trunc({cmap.name},{minval:.2f},{maxval:.2f})", cmap(np.linspace(minval, maxval, n))
        )

    s = matrix[cancer].sort_values(ascending=True)

    norm = plt.Normalize(vmin=s.min(), vmax=s.max())
    cmap_obj = truncate_colormap(plt.get_cmap(cmap), minval=cmap_minval, maxval=cmap_maxval)
    colors = [zero_color if v == 0 else cmap_obj(norm(v)) for v in s]

    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.barh(s.index, s.values, color=colors, edgecolor="white", linewidth=0.5)

    for bar, val in zip(bars, s.values):
        ax.text(
            bar.get_width() + s.max() * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}",
            va="center",
            ha="left",
            fontsize=annot_fontsize,
            color="dimgray",
        )

    sm = plt.cm.ScalarMappable(cmap=cmap_obj, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label("Median aggregated |SurvSHAP(t)|", fontsize=11)
    cbar.ax.tick_params(labelsize=9)

    ax.set_xlabel("Median aggregated |SurvSHAP(t)|", fontsize=13)
    ax.set_title(
        f"Feature importance for {cancer.replace('_', ' ').title()} Cancer", fontsize=17, pad=15
    )
    ax.set_xlim(0, s.max() * 1.18)
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=1000)

In [ ]:
for cancer in df_all_cohorts.columns:
    cancer_name = [key for key, value in cancer_col_name_map.items() if value == cancer][0]
    plot_cancer_bar(
        df_all_cohorts,
        cancer,
        save_path=GRAPH_PATH / f"{cancer_name}" / f"{cancer_name}_feature_importance.png",
    )

In [ ]:
def plot_heatmap_raw(
    matrix: pd.DataFrame,
    figsize: tuple = (15, 8),
    annot_fmt: str = ".2f",
    cmap: str = "Blues",
    cmap_minval: float = 0.3,
    cmap_maxval: float = 1.0,
    save_path=None,
):
    def truncate_colormap(cmap, minval=0.3, maxval=1.0, n=256):
        import matplotlib.colors as mcolors
        import numpy as np

        return mcolors.LinearSegmentedColormap.from_list(
            f"trunc({cmap.name},{minval:.2f},{maxval:.2f})", cmap(np.linspace(minval, maxval, n))
        )

    row_order = matrix.mean(axis=1).sort_values(ascending=False).index
    m = matrix.loc[row_order]
    m = m.reindex(sorted(m.columns), axis=1)

    cmap_obj = truncate_colormap(plt.get_cmap(cmap), minval=cmap_minval, maxval=cmap_maxval)

    fig, ax = plt.subplots(figsize=figsize)

    im = sns.heatmap(
        m,
        annot=True,
        fmt=annot_fmt,
        cmap=cmap_obj,
        cbar_kws={"label": "Median aggregated SurvSHAP(t)"},
        linewidths=0.5,
        linecolor="white",
        ax=ax,
    )
    im.collections[0].colorbar.set_label("Median aggregated SurvSHAP(t)", fontsize=12)

    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title("Absolute feature importance across cancer cohorts", fontsize=20, pad=15)
    plt.xticks(rotation=35, ha="right", fontsize=12)
    plt.yticks(rotation=0, fontsize=12)
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=1000)

In [ ]:
plot_heatmap_raw(df_all_cohorts, save_path=GRAPH_PATH / "absolute_feature_importance.png")

In [ ]:
def plot_heatmap_normalized(
    matrix: pd.DataFrame,
    figsize: tuple = (15, 8),
    cmap: str = "Blues",
    cmap_minval: float = 0.3,
    cmap_maxval: float = 1.0,
    save_path=None,
):
    def truncate_colormap(cmap, minval=0.3, maxval=1.0, n=256):
        import matplotlib.colors as mcolors
        import numpy as np

        return mcolors.LinearSegmentedColormap.from_list(
            f"trunc({cmap.name},{minval:.2f},{maxval:.2f})", cmap(np.linspace(minval, maxval, n))
        )

    col_sums = matrix.sum(axis=0).replace(0, np.nan)
    normalized = matrix.div(col_sums, axis=1)

    row_order = normalized.mean(axis=1).sort_values(ascending=False).index
    normalized = normalized.loc[row_order]
    normalized = normalized.reindex(sorted(normalized.columns), axis=1)

    cmap_obj = truncate_colormap(plt.get_cmap(cmap), minval=cmap_minval, maxval=cmap_maxval)

    fig, ax = plt.subplots(figsize=figsize)
    im = sns.heatmap(
        normalized,
        annot=True,
        fmt=".2f",
        cmap=cmap_obj,
        cbar_kws={"label": "Within-cohort proportional importance"},
        linewidths=0.5,
        linecolor="white",
        vmin=0,
        vmax=normalized.max().max(),
        ax=ax,
    )
    im.collections[0].colorbar.set_label("Within-cohort proportional importance", fontsize=12)

    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title("Relative feature importance within cancer cohorts", fontsize=20, pad=15)
    plt.xticks(rotation=35, ha="right", fontsize=12)
    plt.yticks(rotation=0, fontsize=12)
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=1000)

In [ ]:
plot_heatmap_normalized(df_all_cohorts, save_path=GRAPH_PATH / "relative_feature_importance.png")

In [ ]:
def plot_top_features_grid(
    matrix: pd.DataFrame,
    top_k: int = 5,
    n_cols: int = 5,
    figsize: tuple = (20, 8),
    cmap: str = "Blues",
    cmap_minval: float = 0.3,
    cmap_maxval: float = 1.0,
    save_path=None,
):
    def truncate_colormap(cmap, minval=0.3, maxval=1.0, n=256):
        import matplotlib.colors as mcolors
        import numpy as np

        return mcolors.LinearSegmentedColormap.from_list(
            f"trunc({cmap.name},{minval:.2f},{maxval:.2f})", cmap(np.linspace(minval, maxval, n))
        )

    cohorts = sorted(matrix.columns)
    n_rows = int(np.ceil(len(cohorts) / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize, sharex=False)
    axes = axes.flatten()

    cmap_obj = truncate_colormap(plt.get_cmap(cmap), minval=cmap_minval, maxval=cmap_maxval)

    for i, cohort in enumerate(cohorts):
        ax = axes[i]
        col = matrix[cohort].sort_values(ascending=False).head(top_k)

        norm = plt.Normalize(vmin=col.min(), vmax=col.max())
        colors = [cmap_obj(norm(v)) for v in col.values]

        ax.barh(range(len(col)), col.values, color=colors)
        ax.set_yticks(range(len(col)))
        ax.set_yticklabels(col.index, fontsize=12)
        ax.set_xlabel("MA-SurvSHAP(t)")
        ax.invert_yaxis()
        ax.set_title(cohort.replace("_", " ").title(), fontsize=15)
        ax.tick_params(axis="x", labelsize=10)
        ax.spines[["top", "right"]].set_visible(False)

    for j in range(len(cohorts), len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(
        f"Top-{top_k} features per cohort (by Median Aggregated |SurvSHAP(t)| (MA-SurvSHAP(t)) values",
        fontsize=20,
    )
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=1000)

In [ ]:
plot_top_features_grid(df_all_cohorts, save_path=GRAPH_PATH / "top_features_survshap.png")